In [2]:
import hydra
from omegaconf import DictConfig
from hydra import initialize, compose

import pandas as pd
import os

In [3]:
# Initialize the Hydra config within Jupyter
initialize(config_path="")  # Point to your config directory

# Compose the configuration
cfg = compose(config_name="config")   # Load your main config.yaml

cfg

/tmp/ipykernel_2535039/3567310962.py:2: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="")  # Point to your config directory


{'validation': {'if_validate': False, 'batch_size': 8, 'save_model_every_n_epochs': 250, 'check_val_every_n_epochs': 100, 'check_val_monitor': 'val_loss/absolute_rssd', 'save_top_k_models': 40, 'early_stopping': True, 'early_stopping_patience': 100}, 'test': {'save_dir': '/mlbio_scratch/anagupta/luna/LUNA/exp/test_results', 'checkpoints_parent_dir': '/mlbio_scratch/anagupta/luna/LUNA/exp/checkpoints', 'checkpoints_name_list': 'all', 'batch_size': 1, 'epoch_index': None, 'checkpoint_path': None, 'test_save_parent_path': None, 'checkpoint_name': 'all'}, 'general': {'name': 'luna_2025-05-25_23:10:19', 'wandb': 'online', 'mode': 'train_and_test', 'seed': 0, 'enable_progress_bar': True, 'local_saved_path': '/mlbio_scratch/anagupta/luna/LUNA/exp', 'debug': False}, 'train': {'n_epochs': 500, 'batch_size': 4, 'lr': 0.0005, 'fast_dev_run': False, 'weight_decay': 1e-12}, 'model': {'n_layers': 8, 'diffusion_noise_schedule': 'cosine', 'diffusion_steps': 1000, 'cell_image_encoder': 'CNN', 'nu': {'p

In [4]:
cfg.model.cell_image_encoder

'CNN'

In [5]:


cfg.general.name = 'luna_cnn_test_only' # the name of the experiment
cfg.general.mode = 'test_only' # the mode of the experiment

cfg.test.save_dir = '/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/test_results' # the path to save the test results
cfg.test.checkpoints_parent_dir = '/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/checkpoints' # the path to the parent directory of the checkpoints
cfg.test.checkpoint_name = ['epoch=499.ckpt'] # the name list of the checkpoints

In [ ]:
import torch 

checkpoint = torch.load('/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/checkpoints/epoch=499.ckpt')
# print(checkpoint)
print("Checkpoint keys:")
for key in checkpoint['state_dict'].keys():
    if key.startswith('model.encoder'):
        print(key)


In [6]:
checkpoints_name_list=['epoch=499.ckpt']
gpus_per_node=[1]

In [7]:
!python3 ../main.py general.name='luna_cnn_test_only' general.mode='test_only' test.save_dir='/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/test_results' test.checkpoints_parent_dir='/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/checkpoints' test.checkpoint_name=checkpoints_name_list dataset.maximum_graph_size.test=2000

Seed set to 0
{'validation': {'if_validate': False, 'batch_size': 8, 'save_model_every_n_epochs': 250, 'check_val_every_n_epochs': 100, 'check_val_monitor': 'val_loss/absolute_rssd', 'save_top_k_models': 40, 'early_stopping': True, 'early_stopping_patience': 100}, 'test': {'save_dir': '/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/test_results', 'checkpoints_parent_dir': '/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51/checkpoints', 'checkpoints_name_list': 'all', 'batch_size': 1, 'epoch_index': None, 'checkpoint_path': None, 'test_save_parent_path': None, 'checkpoint_name': 'checkpoints_name_list'}, 'general': {'name': 'luna_cnn_test_only', 'wandb': 'online', 'mode': 'test_only', 'seed': 0, 'enable_progress_bar': True, 'local_saved_path': '/mlbio_scratch/anagupta/luna/outputs/2025-05-25/23-20-51', 'debug': False}, 'train': {'n_epochs': 500, 'batch_size': 4, 'lr': 0.0005, 'fast_dev_run': False, 'weight_decay': 1e-12}, 'model': {'n_layers': 8, 'diffusion_noise_schedu

In [9]:
import torch
torch.cuda.device_count()

8

In [9]:

## The following is the configuration file that you need to use for the experiment. Here is only for information. The command to run the experiment is in the next cell.

from datetime import datetime

# Get current date and time
now = datetime.now()

# Format it as a string
timestamp_str = now.strftime("%Y-%m-%d_%H:%M:%S")

cfg.general.name = 'luna' + '_' + timestamp_str
train_test_data_folder = 'train_test_split_1'

cfg.distribute.gpus_per_node=[1]
# cfg.general.wandb='disabled'
cfg.general.debug = True

cfg.dataset.maximum_graph_size.train=1000
cfg.dataset.maximum_graph_size.test=1000
cfg.train.batch_size=4
cfg.model.hidden_dims.cell_image_dimensions=256
cfg.model.hidden_dims.num_heads=16

# Use the setting to quickly check the model
cfg.validation.check_val_every_n_epochs=1
cfg.validation.save_model_every_n_epochs=1
cfg.train.n_epochs=4
cfg.model.diffusion_steps=2

cfg.dataset.gene_columns_start = 13
cfg.dataset.gene_columns_end = 360


run_directory = '/home/anagupta/luna/runs' + '/' + 'run_'+ timestamp_str
if not os.path.exists(run_directory):
    os.makedirs(run_directory)
cfg.general.local_saved_path = run_directory + '/train_results'
cfg.test.save_dir = run_directory + '/test_results' # Change this to the directory where you want to save the results

data_directory = '/home/anagupta/luna/' + train_test_data_folder
cfg.dataset.train_data_path = data_directory + '/train_data.csv' # Change this to the path of the train csv file
cfg.dataset.test_data_path = data_directory + '/test_data.csv' # Change this to the path of the test csv file
cfg.dataset.slice_images_path = data_directory + '/slice_images' # Change this to the path of the slice images
cfg.dataset.train_cell_images_path = data_directory + '/train_cell_images' # Change this to the path of the train cell images
cfg.dataset.test_cell_images_path = data_directory + '/test_cell_images' # Change this to the path of the test cell images

cfg.dataset.dataset_name = 'luna_with_cell_images' if cfg.dataset.train_cell_images_path else 'luna_without_cell_images'

from omegaconf import OmegaConf

# Save the cfg configuration file
OmegaConf.save(cfg, run_directory + '/config.yaml')

In [5]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.6


In [11]:
HYDRA_FULL_ERROR=1  
output_path =  root_directory + '/output.txt'
!python3 /home/anagupta/luna/LUNA/main.py --config-path=$root_directory --config-name=config.yaml > $output_path

Seed set to 0
/home/anagupta/luna/LUNA/datasets/data_module.py:106: FutureWarning: Index.is_numeric is deprecated. Use pandas.api.types.is_any_real_numeric_dtype instead
  if not self.input_data.index.is_numeric():
/home/anagupta/luna/.venv/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
/home/anagupta/luna/LUNA/datasets/data_module.py:106: FutureWarning: Index.is_numeric is deprecated. Use pandas.api.types.is_any_real_numeric_dtype instead
  if not self.input_data.index.is_numeric():
/home/anagupta/luna/.venv/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:300: Use

In [14]:
import tarfile
from PIL import Image
from io import BytesIO
import os
import numpy as np

extracted_folder = '/home/anagupta/luna/preprocessed_images/cell_imagess'

extracted_images = []

# loop through the extracted folder and get all the tar files
for root, dirs, files in os.walk(extracted_folder):
    for file in files:
        if file.endswith('.tar'):
            tar_file_path = os.path.join(root, file)
            # print(tar_file_path)
            # Open the tar file
            with tarfile.open(tar_file_path, 'r') as tar:
                # Loop through the files in the tar archive
                for tarinfo in tar:
                    # Check if the file inside the tar is an image
                    # print(tarinfo.name)
                    if tarinfo.name.endswith(('.npy')):
                        # Extract the file as a BytesIO object (in memory)
                        file_obj = tar.extractfile(tarinfo)
                        image_data = file_obj.read()
                        # Convert the image data to a PIL Image
                        image_array = np.load(BytesIO(image_data))
                        extracted_images.append(image_array)

print("Number of images extracted:", len(extracted_images))


Number of images extracted: 53912


In [8]:
a = False

if a: 
    b = 1

if b: 
    print("b is true")
else:
    print("b is false")

NameError: name 'b' is not defined